# Phase 1 -- PersonaPlex Emotional Dataset Collection

**PersonaPlex in-house emotion network project** -- this notebook collects the training dataset for the unified emotion head by **running PersonaPlex itself** and capturing its internal representations while it generates emotional replies.

## What this notebook does

1. Verifies the environment, repository, and dependencies (Phase 0 should already be done -- its `phase0_results.json` tells us which representations matter).
2. Downloads the PersonaPlex model assets: the community 4-bit checkpoint (`brianmatzelle/personaplex-7b-v1-bnb-4bit`, ~7 GB, ships its own `moshi` package) plus the gated NVIDIA assets (Mimi codec, text tokenizer, voice-prompt embeddings) -- resumable, checksum-aware, no re-downloads on restart.
3. Instantiates the **production PersonaPlex engine** (`liveTry.py::MoshiOnlyEngine`) with the **exact live-server configuration** (4-bit weights, 8 codebooks, cfg_coef=1.0), extended with the same CUDA-graph-compatible hidden-state capture the live AH server uses (`lm_gen._step` patch returning `(tokens, transformer_out, layer_hidden)`).
4. Drives conversations across a configurable matrix of **voices x scenarios x user-prompt clips** (CREMA-D emotional speech as user turns; scenario text prompts steer the persona). For every generated step it captures: Helium hidden state [4096], 8 Mimi audio codebook tokens, text token, and decoded reply PCM.
5. **Verifies alignment per session, never assumed**: codes->PCM re-decode consistency, hidden<->audio energy cross-correlation, text-token transcript vs ASR of the reply audio.
6. Labels every analysis window with **both teachers, kept separate** (vocal: emotion2vec on the reply audio; semantic: ASR + text classifier), preserves full score vectors, and routes disagreements/low-confidence rows to a **human review queue** with a corrections workflow.
7. Packs resumable per-session `.pt` files + a `windows_index.csv` (100-step = 8 s windows matching the live deque contract) + `phase1_dataset.json` manifest for Phase 2.

## Key principles

- **Intended emotion is metadata, never ground truth.** Scenarios and prompts *steer*; the teachers + human review decide what was *expressed*. Intended->expressed mismatch is measured, not hidden.
- **The live capture contract is replicated exactly** -- the same engine class, the same capture mechanism, the same window geometry (100 steps at 12.5 Hz) the live deque uses.
- **Everything resumable**: asset downloads, session generation, labeling, and window extraction all skip completed work.
- **Multiple voices** (voice-prompt embeddings) so Phase 2 can test speaker/voice generalization with leakage-safe splits.

## Hardware / time / storage

| Item | Requirement |
|---|---|
| GPU | >= 24 GB VRAM recommended (engine ~10 GB 4-bit + teachers ~2-3 GB) |
| Disk | ~15 GB assets + ~1-3 GB per 100 sessions (fp16 hidden states dominate) |
| Time | ~15-25 s per session (voice-prompt reload + ~12 s generation); ~2-3 h for 400 sessions |
| Access | `HF_TOKEN` env var with access to the **gated** `nvidia/personaplex-7b-v1` repo (Mimi, tokenizer, voices) |

## Outputs

| Artifact | Path (under `/workspace/persona_emotion/phase1/`) |
|---|---|
| Packed sessions | `sessions/<session_id>.pt` + `sessions_index.csv` |
| Analysis windows | `windows/` + `windows_index.csv` (dual labels, confidences, agreement flags, voice/scenario groups) |
| Human review | `review_queue.csv` + `corrections.csv` |
| Manifest for Phase 2 | `phase1_dataset.json` |
| Alignment reports | `alignment_report.csv`, `plots/` |

In [ ]:
# ============================ Configuration =================================
import os, sys, json, time, math, random, hashlib, subprocess, shutil, contextlib, types
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/MoshiHead/IMTalker_Personaplex_emotion_v1_auto_emotion_from_helium.git"

WORKSPACE = Path(os.environ.get("WORKSPACE", "/workspace"))
PROJECT_ROOT = Path(os.environ.get("SPEECH2AVATAR_ROOT", str(WORKSPACE / "speech2avatar")))
PHASE0_DIR = Path(os.environ.get("PERSONA_EMO_PHASE0_DIR", str(WORKSPACE / "persona_emotion" / "phase0")))
PHASE1_DIR = Path(os.environ.get("PERSONA_EMO_PHASE1_DIR", str(WORKSPACE / "persona_emotion" / "phase1")))

# --- Model assets ---
BNB4_HF_REPO = "brianmatzelle/personaplex-7b-v1-bnb-4bit"   # community 4-bit checkpoint + its own moshi package
GATED_HF_REPO = "nvidia/personaplex-7b-v1"                  # gated: mimi, tokenizer, voices (needs HF_TOKEN)
BNB4_DIR = Path(os.environ.get("PERSONAPLEX_BNB4_DIR", str(WORKSPACE / "personaplex-7b-v1-bnb-4bit")))

# --- Live-server contract (DO NOT change unless the live server changes) ---
QUANTIZE_4BIT = True
NUM_CODEBOOKS = 8
CFG_COEF = 1.0
DEVICE = "cuda"
TARGET_SR = 24000
FRAME_SIZE = 1920            # 80 ms at 24 kHz == one Mimi/Helium step
STEP_HZ = 12.5
WINDOW_STEPS = 100           # 8 s -- matches the live helium deque window
WINDOW_STRIDE = 50           # 50% overlap
MIN_REPLY_STEPS = 38         # discard sessions whose reply is shorter than ~3 s
LOOKAHEAD_STEPS = 6          # 0.48 s -- documented for Phase 2 live-latency accounting

# --- Collection plan ---
VOICE_PROMPTS = ["NATF0.pt", "NATF1.pt", "NATM0.pt", "NATM1.pt"]   # intersected with available voices at runtime
TARGET_SESSIONS = 120        # total sessions across the voice x scenario x clip matrix
MAX_REPLY_SEC = 12.0         # generation cap per session
USER_CLIP_SEC = 4.0          # user turn length fed to the model

# --- Teachers (offline labeling; identical contract to Phase 0) ---
AUDIO_TEACHER_MODEL = "emotion2vec/emotion2vec_plus_base"
TEXT_TEACHER_MODEL = "j-hartmann/emotion-english-distilroberta-base"
ASR_MODEL = "openai/whisper-tiny.en"
RUN_TEXT_TEACHER = True
TEACHER_DEVICE = "cuda"      # set "cpu" if VRAM is tight
EMOTION_NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}
EMOTION_IDS = {v: k for k, v in EMOTION_NAMES.items()}
NUM_EMOTIONS = 6
DISGUST_MAP = 3
SEED = 2027

# --- Derived paths ---
SESSIONS_DIR = PHASE1_DIR / "sessions"
WINDOWS_DIR = PHASE1_DIR / "windows"
PLOTS_DIR = PHASE1_DIR / "plots"
ASSETS_DIR = PHASE1_DIR / "assets"
SESSIONS_INDEX_CSV = PHASE1_DIR / "sessions_index.csv"
WINDOWS_INDEX_CSV = PHASE1_DIR / "windows_index.csv"
ALIGNMENT_CSV = PHASE1_DIR / "alignment_report.csv"
REVIEW_QUEUE_CSV = PHASE1_DIR / "review_queue.csv"
CORRECTIONS_CSV = PHASE1_DIR / "corrections.csv"
MANIFEST_JSON = PHASE1_DIR / "phase1_dataset.json"
STATE_JSON = PHASE1_DIR / "state.json"

for _p in (PHASE1_DIR, SESSIONS_DIR, WINDOWS_DIR, PLOTS_DIR, ASSETS_DIR):
    _p.mkdir(parents=True, exist_ok=True)

def log(msg):
    line = "[%s] %s" % (time.strftime("%H:%M:%S"), msg)
    print(line, flush=True)
    with open(PHASE1_DIR / "phase1.log", "a", encoding="utf-8") as f:
        f.write(line + "\n")

def save_json(obj, path):
    tmp = str(path) + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)

random.seed(SEED)
import numpy as np
np.random.seed(SEED)
log("Configuration loaded. PHASE1_DIR = %s" % PHASE1_DIR)
log("Collection plan: %d sessions | voices=%s | window=%d steps @ %.1f Hz" % (TARGET_SESSIONS, VOICE_PROMPTS, WINDOW_STEPS, STEP_HZ))

In [ ]:
# =========================== Environment inspection =========================
def run_cmd(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=90).stdout.strip()
    except Exception as e:
        return "<error: %s>" % e

log("Python : " + sys.version.split()[0])
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required: the PersonaPlex engine runs on GPU (~10 GB VRAM in 4-bit).")
log("GPU    : %s (%.1f GB VRAM)" % (torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 1e9))
log("torch  : %s | CUDA %s" % (torch.__version__, torch.version.cuda))
assert torch.cuda.get_device_properties(0).total_memory / 1e9 >= 15.0, "Need >= ~16 GB VRAM (engine ~10 GB + teachers ~3 GB). Use a larger GPU or set TEACHER_DEVICE='cpu'."
_disk = shutil.disk_usage(str(WORKSPACE if WORKSPACE.exists() else Path.cwd()))
log("Disk   : %.1f GB free" % (_disk.free / 1e9))
assert _disk.free / 1e9 > 25.0, "Need ~25 GB free (model assets ~15 GB + dataset)."
_hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not _hf_token:
    log("WARNING: HF_TOKEN not set. The gated %s assets (mimi, tokenizer, voices) will fail to download. Set HF_TOKEN and re-run." % GATED_HF_REPO)
else:
    log("HF_TOKEN is set (gated asset downloads enabled).")
log("Environment OK.")

In [ ]:
# ========================= Repository acquisition ===========================
REPO_MARKER_FILES = ["IMTalker/liveTry.py", "IMTalker/auto_emotion.py", "personaplex/moshi/moshi/models/lm.py"]

def looks_like_repo(p):
    p = Path(p)
    return p.is_dir() and all((p / f).exists() for f in REPO_MARKER_FILES)

_in_place = None
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if looks_like_repo(_cand):
        _in_place = _cand
        break
if _in_place is None and looks_like_repo(PROJECT_ROOT):
    _in_place = PROJECT_ROOT
if _in_place is not None:
    PROJECT_ROOT = _in_place
    log("Using in-place repository: %s" % PROJECT_ROOT)
elif not looks_like_repo(PROJECT_ROOT):
    log("Cloning %s -> %s" % (GITHUB_REPO_URL, PROJECT_ROOT))
    _r = subprocess.run("git clone --depth 1 %s %s" % (GITHUB_REPO_URL, str(PROJECT_ROOT)), shell=True, capture_output=True, text=True)
    if _r.returncode != 0 or not looks_like_repo(PROJECT_ROOT):
        raise RuntimeError("Repository unavailable: %s. Copy this notebook into your local repository clone and run it there." % _r.stderr[-800:])
REPO_COMMIT = run_cmd("git -C %s rev-parse HEAD" % str(PROJECT_ROOT)) or "unknown"
IMTALKER_DIR = PROJECT_ROOT / "IMTalker"
log("Repository ready (commit %s)" % REPO_COMMIT[:12])

In [ ]:
# ============================ Dependencies ==================================
# Targeted install for Phase 1. liveTry.py imports cv2/fastapi at module level
# (the web server path); the engine itself needs torch/torchaudio/sphn/einops/
# sentencepiece/bitsandbytes (4-bit unpickling) and the moshi package from the
# bnb4 checkpoint directory. Idempotent: installed packages are skipped.

def pip_install(pkg_spec):
    r = subprocess.run("%s -m pip install --quiet %s" % (sys.executable, pkg_spec), shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2500:]); print(r.stderr[-2500:])
        raise RuntimeError("pip install failed: %s" % pkg_spec)
    log("pip ok: %s" % pkg_spec)

def have(mod):
    return subprocess.run("%s -c \"import %s\"" % (sys.executable, mod), shell=True, capture_output=True).returncode == 0

for _m, _pkg in [
    ("cv2", "opencv-python"),
    ("fastapi", "fastapi"),
    ("bitsandbytes", "bitsandbytes"),
    ("sphn", "sphn>=0.2.0,<0.3.0"),
    ("einops", "einops"),
    ("sentencepiece", "sentencepiece"),
    ("safetensors", "safetensors"),
    ("soundfile", "soundfile"),
    ("torchaudio", "torchaudio"),
    ("sklearn", "scikit-learn==1.3.2"),
    ("tqdm", "tqdm"),
    ("requests", "requests"),
]:
    if not have(_m):
        pip_install(_pkg)
if not have("funasr"):
    pip_install("funasr")
pip_install("\"numpy<2.0.0\" \"transformers==4.30.2\" \"pandas<2.2.0\" \"matplotlib<3.9.0\"")
log("All dependencies ready.")

In [ ]:
# ============================ Model assets ==================================
# 1) bnb4 checkpoint (public): the 4-bit PersonaPlex weights + its own moshi
#    package (the engine imports moshi from THIS directory, exactly like the
#    live server: _ensure_moshi_importable(moshi_root)).
# 2) Gated NVIDIA assets: mimi codec, text tokenizer, voices.tgz -- downloaded
#    lazily by the engine via hf_hub_download; we pre-verify access here.
# Everything is cached by huggingface_hub; re-runs do not re-download.
from huggingface_hub import hf_hub_download, list_repo_files

BNB4_FILES = ["model_bnb_4bit.pt"]   # ~6.98 GB; the moshi/ package ships inside the repo too

def ensure_bnb4():
    BNB4_DIR.mkdir(parents=True, exist_ok=True)
    marker = BNB4_DIR / ".complete"
    if marker.exists() and (BNB4_DIR / "model_bnb_4bit.pt").exists() and (BNB4_DIR / "moshi" / "moshi" / "models" / "lm.py").exists():
        log("bnb4 checkpoint already present: %s" % BNB4_DIR)
        return
    files = list_repo_files(BNB4_HF_REPO)
    log("downloading %d files from %s ..." % (len(files), BNB4_HF_REPO))
    for f in files:
        dst = BNB4_DIR / f
        if dst.exists():
            continue
        log("  %s" % f)
        src = hf_hub_download(BNB4_HF_REPO, f)
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
    marker.write_text("ok")
    log("bnb4 checkpoint ready.")

ensure_bnb4()
assert (BNB4_DIR / "model_bnb_4bit.pt").exists()
assert (BNB4_DIR / "moshi" / "moshi" / "models" / "lm.py").exists(), "bnb4 repo must ship its own moshi package"

# gated assets: verify access + resolve local paths (engine also accepts these)
MIMI_PATH = hf_hub_download(GATED_HF_REPO, "tokenizer-e351c8d8-checkpoint125.safetensors", token=_hf_token)
TOKENIZER_PATH = hf_hub_download(GATED_HF_REPO, "tokenizer_spm_32k_3.model", token=_hf_token)
log("mimi: %s" % MIMI_PATH)
log("tokenizer: %s" % TOKENIZER_PATH)
# voices.tgz is resolved lazily by engine._resolve_voice_prompt_path(); pre-fetch
# it here so we can validate the configured voice list before collecting.
_voices_tgz = Path(hf_hub_download(GATED_HF_REPO, "voices.tgz", token=_hf_token))
VOICES_DIR = _voices_tgz.parent / "voices"
if not VOICES_DIR.exists():
    import tarfile
    with tarfile.open(_voices_tgz, "r:gz") as tar:
        tar.extractall(path=_voices_tgz.parent)
_available_voices = sorted(p.name for p in VOICES_DIR.rglob("*.pt"))
log("available voice prompts: %s" % _available_voices)
VOICE_PROMPTS = [v for v in VOICE_PROMPTS if v in _available_voices] or _available_voices[:2]
log("using voices: %s" % VOICE_PROMPTS)
MOSHI_WEIGHT = str(BNB4_DIR / "model_bnb_4bit.pt")

## Capture engine

`EmotionCaptureEngine` subclasses the production `liveTry.py::MoshiOnlyEngine` and adds hidden-state capture using **exactly the mechanism the live AH server uses in production**: patch `lm_gen._step` with a wrapper that calls `prepare_step_input` -> `state.graphed_main` -> `process_transformer_output` and returns `(tokens, transformer_out, layer_hidden)`. CUDA graph replay stays enabled (no Python forward hooks in the hot path).

Per generated step we capture, all step-aligned by construction (they come from the same `lm_out` tuple):

- `helium` [4096] -- layer[-2] main-LM hidden state (post-transformer, pre-out_norm), fp16
- `codes` [8] -- the 8 reply audio codebook tokens
- `text_token` -- the reply text token
- `reply_pcm` [1920] -- Mimi-decoded reply audio for this step (80 ms)

The engine configuration is the **live-server contract**: 4-bit weights, `num_codebooks=8`, `cfg_coef=1.0`, same gated assets. If this notebook runs, the collected data matches what the live system produces bit-for-bit at the representation level.

In [ ]:
# ===================== Capture engine (production-faithful) ==================
if str(IMTALKER_DIR) not in sys.path:
    sys.path.insert(0, str(IMTALKER_DIR))
from liveTry import MoshiOnlyEngine, TARGET_SR as _LT_SR, FRAME_SIZE as _LT_FRAME
assert _LT_SR == TARGET_SR and _LT_FRAME == FRAME_SIZE

class EmotionCaptureEngine(MoshiOnlyEngine):
    def __init__(self, *args, **kwargs):
        self.capture = []          # per-step records; None entries keep step indexing
        self.capture_enabled = True
        super().__init__(*args, **kwargs)
        self._install_hidden_capture()

    def _install_hidden_capture(self):
        lm_gen = self.lm_gen
        if not (hasattr(lm_gen, "prepare_step_input") and hasattr(lm_gen, "process_transformer_output")):
            raise RuntimeError("This moshi build lacks the prepare_step_input/process_transformer_output API used by the live AH server; cannot capture hidden states with CUDA graphs intact.")

        @torch.no_grad()
        def personaplex_step_with_hidden(self_gen, input_tokens=None, moshi_tokens=None, text_token=None, depformer_replace_tokens=None):
            prepared = self_gen.prepare_step_input(input_tokens, moshi_tokens, text_token)
            if prepared is None:
                return None
            input_, provided_, target_, model_input_position, target_position = prepared
            state = self_gen._streaming_state
            transformer_out, text_logits = state.graphed_main(input_)
            output = self_gen.process_transformer_output(
                transformer_out, text_logits, provided_, target_, model_input_position, target_position)
            return output, transformer_out, transformer_out

        lm_gen._step = types.MethodType(personaplex_step_with_hidden, lm_gen)
        lm_gen.streaming_forever(1)
        self._warmup_runtime()
        log("installed PersonaPlex graphed hidden capture (layer[-2], live-server mechanism)")

    @torch.no_grad()
    def _step(self, pcm24: np.ndarray) -> dict:
        self.step += 1
        chunk = torch.from_numpy(pcm24).to(self.device, dtype=torch.float32)[None, None]
        codes = self.mimi.encode(chunk)
        if self.skip_first:
            self.mimi.reset_streaming()
            self.skip_first = False
        lm_out = self.lm_gen._step(codes[:, :, :1])
        if lm_out is None:
            if self.capture_enabled:
                self.capture.append(None)
            return {"step": self.step, "reply_pcm": np.zeros(FRAME_SIZE, dtype=np.float32), "token": -1, "piece": ""}
        if not (isinstance(lm_out, tuple) and len(lm_out) == 3):
            raise RuntimeError("layer[-2] capture contract failure: got %s" % type(lm_out))
        tokens, _transformer_out, layer_hidden = lm_out
        text_token = int(tokens[0, 0, 0].detach().item())
        piece = self.decode_piece(text_token)
        if piece:
            self.audio_text += piece
        reply = self.mimi.decode(tokens[:, 1:])
        reply_pcm = reply[0, 0].detach().float().cpu().numpy()
        if reply_pcm.shape[0] < FRAME_SIZE:
            reply_pcm = np.pad(reply_pcm, (0, FRAME_SIZE - reply_pcm.shape[0]))
        elif reply_pcm.shape[0] > FRAME_SIZE:
            reply_pcm = reply_pcm[:FRAME_SIZE]
        if self.capture_enabled:
            self.capture.append({
                "helium": layer_hidden[:1, -1:].detach().to("cpu", dtype=torch.float16)[0, 0],  # [4096]
                "codes": tokens[:, 1:].detach().to("cpu", copy=True)[0, :, 0],                  # [8]
                "text_token": text_token,
                "reply_pcm": reply_pcm.astype(np.float16),
            })
        return {"step": self.step, "reply_pcm": reply_pcm, "token": text_token, "piece": piece}

    def drive(self, pcm24: np.ndarray) -> list:
        n_frames = int(np.ceil(len(pcm24) / FRAME_SIZE))
        padded = np.pad(pcm24, (0, n_frames * FRAME_SIZE - len(pcm24)))
        events = []
        for i in range(n_frames):
            events.append(self._step(padded[i * FRAME_SIZE:(i + 1) * FRAME_SIZE]))
        return events

log("EmotionCaptureEngine defined (subclass of the production MoshiOnlyEngine).")

In [ ]:
# ================= Engine instantiation + sanity capture =====================
# Exact live-server contract: 4-bit weights, 8 codebooks, cfg_coef=1.0, same
# gated assets. First instantiation compiles CUDA graphs; allow a few minutes.
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
engine = EmotionCaptureEngine(
    moshi_root=str(BNB4_DIR),
    mimi_hf_repo=GATED_HF_REPO,
    device=DEVICE,
    cfg_coef=CFG_COEF,
    placeholder_jpeg_b64="",
    moshi_weight=MOSHI_WEIGHT,
    mimi_weight=MIMI_PATH,
    tokenizer=TOKENIZER_PATH,
    quantize_4bit=QUANTIZE_4BIT,
    num_codebooks=NUM_CODEBOOKS,
    voice_prompt=VOICE_PROMPTS[0],
    voice_prompt_dir=str(VOICES_DIR),
    text_prompt="You are a friendly, expressive conversational assistant.",
)
ENGINE_VRAM_GB = torch.cuda.max_memory_allocated() / 1e9
log("engine ready in %.1f s | peak VRAM %.2f GB" % (time.time() - t0, ENGINE_VRAM_GB))

# sanity capture: 3 s of silence; verify the capture contract shape-by-shape
engine.capture_enabled = True
engine.capture = []
engine.reset_session()
engine.capture = []   # reset_session re-applies prompts; capture only the driven frames
_events = engine.drive(np.zeros(3 * TARGET_SR, dtype=np.float32))
_recs = [c for c in engine.capture if c is not None]
log("sanity capture: %d steps driven, %d non-None captures" % (len(engine.capture), len(_recs)))
assert len(_recs) > 0, "no captures -- the hidden-capture patch is not engaged"
_h, _c = _recs[0]["helium"], _recs[0]["codes"]
assert tuple(_h.shape) == (4096,) and _h.dtype == torch.float16, "unexpected helium shape/dtype: %s %s" % (_h.shape, _h.dtype)
assert tuple(_c.shape) == (NUM_CODEBOOKS,), "unexpected codes shape: %s" % tuple(_c.shape)
assert _recs[0]["reply_pcm"].shape[0] == FRAME_SIZE
log("capture contract verified: helium [4096] fp16 | codes [8] | reply_pcm [1920] per step")
log("model is alive: sample text token ids %s" % [r["text_token"] for r in _recs[:8]])

## Per-session alignment verification

Alignment is **verified, never assumed**, at three levels for every session:

1. **codes -> PCM consistency** -- re-decode the saved reply codes through a fresh Mimi streaming context and compare with the saved reply PCM (max abs error; catches capture/save bugs).
2. **hidden <-> audio energy** -- cross-correlation between per-step hidden-state norms and per-step reply PCM RMS must peak at lag 0 (+/-1 step allowed for Mimi codebook delays).
3. **text tokens <-> audio** -- detokenize saved text tokens and compare with an ASR transcript of the reply audio (normalized word overlap); catches token/audio stream drift.

Sessions failing any check are flagged in `alignment_report.csv` and excluded from window extraction (but kept on disk for debugging).

In [ ]:
# ================= Per-session alignment verification ========================
def detokenize_text_tokens(token_ids):
    pieces = []
    for t in token_ids:
        if t in (0, 3, -1):
            continue
        with contextlib.suppress(Exception):
            pieces.append(engine.tokenizer.id_to_piece(int(t)))
    text = "".join(pieces).replace("\u2581", " ").strip()
    return text

@torch.no_grad()
def verify_codes_pcm(session):
    codes = session["codes"]                    # [8, T] long
    pcm_ref = session["reply_pcm"]              # [T*1920] float
    T = codes.shape[1]
    fresh = engine.mimi
    fresh.reset_streaming()
    outs = []
    chunk = min(T, 50)
    for i in range(0, T, chunk):
        seg = codes[:, i:i + chunk].unsqueeze(0).to(engine.device)
        outs.append(fresh.decode(seg)[0, 0].detach().float().cpu().numpy())
    pcm_re = np.concatenate(outs)[:pcm_ref.shape[0]]
    n = min(len(pcm_re), len(pcm_ref))
    err = float(np.max(np.abs(pcm_re[:n].astype(np.float32) - pcm_ref[:n].astype(np.float32))))
    return err

def verify_hidden_energy(session):
    h = session["helium"].float()               # [T, 4096]
    pcm = session["reply_pcm"].astype(np.float32)
    T = h.shape[0]
    idx = np.linspace(0, len(pcm), T + 1).astype(int)
    step_rms = np.array([np.sqrt(np.mean(pcm[idx[i]:idx[i + 1]] ** 2) + 1e-12) for i in range(T)])
    norms = h.norm(dim=-1).numpy()
    a = (norms - norms.mean()) / (norms.std() + 1e-8)
    b = (step_rms - step_rms.mean()) / (step_rms.std() + 1e-8)
    best = (-2.0, 0)
    for lag in range(-2, 3):
        if lag < 0:
            x, y = a[-lag:], b[:len(b) + lag]
        elif lag > 0:
            x, y = a[:-lag], b[lag:]
        else:
            x, y = a, b
        n = min(len(x), len(y))
        if n < 8:
            continue
        c = float(np.mean(x[:n] * y[:n]))
        if c > best[0]:
            best = (c, lag)
    return best

def word_overlap(a, b):
    wa = set(a.lower().split()); wb = set(b.lower().split())
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

def load_session(path):
    s = torch.load(path, map_location="cpu", weights_only=False)
    return s

def verify_session(path, asr_fn=None):
    s = load_session(path)
    rec = {"session_id": s["meta"]["session_id"]}
    try:
        rec["codes_pcm_maxerr"] = verify_codes_pcm(s)
    except Exception as e:
        rec["codes_pcm_maxerr"] = float("nan")
        rec["codes_pcm_error"] = str(e)[:120]
    corr, lag = verify_hidden_energy(s)
    rec["energy_corr"] = corr
    rec["energy_lag"] = lag
    rec["detok_text"] = detokenize_text_tokens(s["text_tokens"].tolist())
    if asr_fn is not None and rec["detok_text"]:
        try:
            rec["asr_text"] = asr_fn(s["reply_pcm"].astype(np.float32))
            rec["text_audio_overlap"] = word_overlap(rec["detok_text"], rec["asr_text"])
        except Exception:
            rec["text_audio_overlap"] = float("nan")
    rec["ok"] = bool(
        (np.isnan(rec.get("codes_pcm_maxerr", np.nan)) or rec["codes_pcm_maxerr"] < 0.35)
        and abs(rec["energy_lag"]) <= 1 and rec["energy_corr"] > 0.05
        and (np.isnan(rec.get("text_audio_overlap", np.nan)) or rec["text_audio_overlap"] > 0.15 or not rec["detok_text"])
    )
    return rec

log("alignment verification functions ready.")

## Scenario, voice, and user-clip design

**Intended vs expressed emotion.** Each session is *steered* toward an intended emotion by (a) the scenario persona text prompt and (b) an emotional user-prompt clip. The intended emotion is **metadata only** -- teachers + human review decide the expressed emotion. This is deliberate: PersonaPlex does not always comply with steering, and the intended->expressed mismatch distribution is itself recorded (it matters for understanding how controllable the model is).

**Voices.** Multiple PersonaPlex voice-prompt embeddings so Phase 2 can split by voice (leakage-safe generalization test).

**User clips.** CREMA-D emotional speech (public GitHub mirror) as the user turn. If downloads fail, drop your own 24 kHz-compatible WAVs into `assets/user_clips/` and re-run -- the collector picks up whatever is present.

In [ ]:
# ================= Scenarios + user clips ====================================
SCENARIOS = [
    {"id": "joy_news",    "intended_emotion": "happy",    "text_prompt": "You are an excited, warm friend. You just heard wonderful news and cannot wait to share your joy. Speak with bright, upbeat energy."},
    {"id": "grief_talk",  "intended_emotion": "sad",      "text_prompt": "You are a gentle, melancholic friend going through a hard time. Speak softly and slowly, with a heavy heart."},
    {"id": "angry_rant",  "intended_emotion": "angry",    "text_prompt": "You are frustrated and fed up after a terrible day. Speak with sharp, irritated energy and let the annoyance show."},
    {"id": "fear_story",  "intended_emotion": "fear",     "text_prompt": "You are anxious and uneasy, as if something frightening just happened. Speak nervously, with worried hesitation."},
    {"id": "surprise_it", "intended_emotion": "surprise", "text_prompt": "You are astonished by an unexpected twist. React with genuine amazement and wide-eyed curiosity."},
    {"id": "calm_neutral","intended_emotion": "neutral",  "text_prompt": "You are a calm, matter-of-fact assistant. Speak evenly and plainly, without strong emotion."},
]

CREMA_BASE = "https://github.com/CheyneyComputerScience/CREMA-D/raw/master/AudioWAV"
CREMA_CLIPS = [
    ("1001_DFA_HAP_XX.wav", "happy"), ("1001_IEO_HAP_HI.wav", "happy"),
    ("1001_DFA_SAD_XX.wav", "sad"),   ("1001_IEO_SAD_HI.wav", "sad"),
    ("1001_DFA_ANG_XX.wav", "angry"), ("1001_IEO_ANG_HI.wav", "angry"),
    ("1001_DFA_FEA_XX.wav", "fear"),  ("1001_IEO_FEA_HI.wav", "fear"),
    ("1001_DFA_NEU_XX.wav", "neutral"),("1001_IEO_NEU_HI.wav", "neutral"),
    ("1001_DFA_DIS_XX.wav", "surprise"),("1001_IEO_DIS_HI.wav", "surprise"),
]

USER_CLIPS_DIR = ASSETS_DIR / "user_clips"
USER_CLIPS_DIR.mkdir(parents=True, exist_ok=True)

def fetch_user_clips():
    import requests, soundfile as sf, torchaudio
    got = []
    for fname, emo in CREMA_CLIPS:
        dst = USER_CLIPS_DIR / fname
        if not dst.exists():
            try:
                r = requests.get(CREMA_BASE + "/" + fname, timeout=60)
                if r.status_code != 200 or len(r.content) < 10000:
                    log("  clip unavailable (%s): HTTP %s" % (fname, r.status_code))
                    continue
                dst.write_bytes(r.content)
            except Exception as e:
                log("  clip download failed (%s): %s" % (fname, str(e)[:120]))
                continue
        got.append((dst, emo))
    # plus any user-provided wavs already in the directory
    for p in sorted(USER_CLIPS_DIR.glob("*.wav")):
        if all(p != g[0] for g in got):
            got.append((p, "unknown"))
    return got

user_clips = fetch_user_clips()
log("user clips available: %d" % len(user_clips))
assert len(user_clips) >= 4, "Need at least 4 user-prompt clips. Provide HF network access or drop WAVs into %s" % USER_CLIPS_DIR

import soundfile as sf, torchaudio

def load_user_clip(path, seconds=USER_CLIP_SEC):
    wav, sr = sf.read(str(path))
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = torchaudio.functional.resample(torch.from_numpy(np.asarray(wav, np.float32).reshape(1, -1)), sr, TARGET_SR)[0].numpy()
    n = int(seconds * TARGET_SR)
    if len(wav) > n:
        peak = int(np.argmax(np.convolve(np.abs(wav), np.ones(n) / n, mode="valid")))
        wav = wav[peak:peak + n]
    return wav

log("scenarios: %d | voices: %d | clips: %d" % (len(SCENARIOS), len(VOICE_PROMPTS), len(user_clips)))

In [ ]:
# ===================== Collection loop (resumable) ===========================
# For each (voice, scenario, clip) combination: reset the session (re-applies
# voice + text prompts exactly like a fresh live connection), drive the user
# clip, then drive silence while PersonaPlex replies. The voiced reply region
# is detected from the captured reply PCM energy; sessions with no/too-short
# replies are recorded as failed and skipped on re-run (no duplicate work).
import pandas as pd
from tqdm.auto import tqdm

def session_paths(sid):
    return SESSIONS_DIR / (sid + ".pt")

def detect_reply_region(recs, pad_steps=4, min_voiced=8):
    rms = np.array([float(np.sqrt(np.mean(np.asarray(r["reply_pcm"], np.float32) ** 2))) for r in recs])
    thr = max(0.01, 0.2 * float(rms.max()))
    voiced = rms > thr
    if voiced.sum() < min_voiced:
        return None
    idxs = np.where(voiced)[0]
    return max(0, int(idxs[0]) - pad_steps), min(len(recs), int(idxs[-1]) + pad_steps)

def collect_session(voice, scenario, clip_path, clip_emo, sid):
    engine.voice_prompt = voice
    engine.text_prompt = scenario["text_prompt"]
    engine.reset_session()
    engine.capture = []
    user_pcm = load_user_clip(clip_path)
    engine.drive(user_pcm)
    n_user = len(engine.capture)
    engine.drive(np.zeros(int(MAX_REPLY_SEC * TARGET_SR), dtype=np.float32))
    recs = [c for c in engine.capture[n_user:] if c is not None]
    region = detect_reply_region(recs)
    if region is None:
        return {"session_id": sid, "status": "no_reply", "voice": voice, "scenario_id": scenario["id"]}
    lo, hi = region
    kept = recs[lo:hi]
    if len(kept) < MIN_REPLY_STEPS:
        return {"session_id": sid, "status": "too_short", "voice": voice, "scenario_id": scenario["id"], "n_steps": len(kept)}
    helium = torch.stack([r["helium"] for r in kept])                    # [T,4096] fp16
    codes = torch.stack([r["codes"] for r in kept]).t().contiguous()     # [8,T]
    text_tokens = torch.tensor([r["text_token"] for r in kept], dtype=torch.long)
    reply_pcm = np.concatenate([np.asarray(r["reply_pcm"], np.float32) for r in kept])
    detok = detokenize_text_tokens(text_tokens.tolist())
    session = {
        "helium": helium, "codes": codes, "text_tokens": text_tokens,
        "reply_pcm": reply_pcm.astype(np.float16),
        "meta": {
            "session_id": sid, "voice": voice, "scenario_id": scenario["id"],
            "intended_emotion": scenario["intended_emotion"],
            "user_clip": Path(clip_path).name, "user_clip_emotion": clip_emo,
            "n_steps": int(helium.shape[0]), "reply_sec": round(helium.shape[0] / STEP_HZ, 2),
            "detok_text": detok, "created": time.strftime("%Y-%m-%d %H:%M:%S"),
            "contract": {"hidden_dim": 4096, "step_hz": STEP_HZ, "num_codebooks": NUM_CODEBOOKS,
                         "quantize_4bit": QUANTIZE_4BIT, "cfg_coef": CFG_COEF},
        },
    }
    torch.save(session, session_paths(sid))
    return {"session_id": sid, "status": "ok", "voice": voice, "scenario_id": scenario["id"],
            "intended_emotion": scenario["intended_emotion"], "user_clip": Path(clip_path).name,
            "user_clip_emotion": clip_emo, "n_steps": int(helium.shape[0]),
            "reply_sec": round(helium.shape[0] / STEP_HZ, 2), "detok_text": detok[:200]}

# build the combination plan (round-robin across voices and clips)
_plan = []
_i = 0
while len(_plan) < TARGET_SESSIONS:
    voice = VOICE_PROMPTS[_i % len(VOICE_PROMPTS)]
    scenario = SCENARIOS[(_i // len(VOICE_PROMPTS)) % len(SCENARIOS)]
    clip_path, clip_emo = user_clips[(_i // (len(VOICE_PROMPTS) * len(SCENARIOS))) % len(user_clips)]
    sid = "s%04d_%s_%s" % (_i, Path(voice).stem, scenario["id"])
    _plan.append((voice, scenario, clip_path, clip_emo, sid))
    _i += 1

if SESSIONS_INDEX_CSV.exists():
    sessions_index = pd.read_csv(SESSIONS_INDEX_CSV)
else:
    sessions_index = pd.DataFrame(columns=["session_id", "status", "voice", "scenario_id", "intended_emotion",
                                           "user_clip", "user_clip_emotion", "n_steps", "reply_sec", "detok_text"])
_done = set(sessions_index.session_id.astype(str)) if len(sessions_index) else set()

log("collection plan: %d sessions (%d already done)" % (len(_plan), len(_done)))
for voice, scenario, clip_path, clip_emo, sid in tqdm(_plan, desc="collecting"):
    if sid in _done:
        continue
    try:
        row = collect_session(voice, scenario, clip_path, clip_emo, sid)
    except Exception as e:
        log("session %s FAILED: %s" % (sid, str(e)[:200]))
        row = {"session_id": sid, "status": "error: %s" % str(e)[:120], "voice": voice, "scenario_id": scenario["id"]}
    sessions_index = pd.concat([sessions_index, pd.DataFrame([row])], ignore_index=True)
    sessions_index.to_csv(SESSIONS_INDEX_CSV, index=False)
    if row["status"] == "ok":
        log("  %s: %d steps (%.1fs) voice=%s scenario=%s" % (sid, row["n_steps"], row["reply_sec"], voice, scenario["id"]))

_ok = sessions_index[sessions_index.status == "ok"]
log("collection pass complete: %d ok / %d total" % (len(_ok), len(sessions_index)))
print(sessions_index.status.value_counts().to_string())
print()
print("sessions by voice:")
print(_ok.voice.value_counts().to_string())
log("peak VRAM: %.2f GB" % (torch.cuda.max_memory_allocated() / 1e9))

In [ ]:
# ============== Alignment verification over collected sessions ===============
# Runs the three-level verification on every collected session. The ASR-based
# text/audio check runs only if RUN_TEXT_TEACHER is enabled (loads whisper lazily).
import pandas as pd
_ok_sessions = sessions_index[sessions_index.status == "ok"].session_id.astype(str).tolist()
log("verifying alignment for %d sessions" % len(_ok_sessions))

_asr_fn = None
if RUN_TEXT_TEACHER:
    from transformers import pipeline as _hf_pipeline
    _asr_pipe = _hf_pipeline("automatic-speech-recognition", model=ASR_MODEL, device=0 if (TEACHER_DEVICE == "cuda" and torch.cuda.is_available()) else -1)
    import torchaudio as _ta
    def _asr_fn(pcm24):
        wav = _ta.functional.resample(torch.from_numpy(pcm24.reshape(1, -1)), TARGET_SR, 16000)[0].numpy()
        return str(_asr_pipe(wav)["text"]).strip()

_rows = []
for _sid in tqdm(_ok_sessions, desc="alignment"):
    try:
        _rows.append(verify_session(session_paths(_sid), asr_fn=_asr_fn))
    except Exception as e:
        _rows.append({"session_id": _sid, "ok": False, "error": str(e)[:150]})
align_df = pd.DataFrame(_rows)
align_df.to_csv(ALIGNMENT_CSV, index=False)
_n_ok = int(align_df.ok.sum()) if "ok" in align_df else 0
log("alignment: %d/%d sessions OK" % (_n_ok, len(align_df)))
if len(align_df):
    print(align_df[[c for c in ("session_id", "codes_pcm_maxerr", "energy_corr", "energy_lag", "text_audio_overlap", "ok") if c in align_df.columns]].describe(include="all").to_string())
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
if "energy_corr" in align_df:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    align_df.energy_corr.dropna().plot.hist(ax=axes[0], bins=30); axes[0].set_title("hidden<->audio energy corr")
    align_df.energy_lag.dropna().plot.hist(ax=axes[1], bins=np.arange(-2.5, 3.5)); axes[1].set_title("energy lag (steps)")
    fig.tight_layout(); fig.savefig(str(PLOTS_DIR / "alignment_sessions.png"), dpi=110); plt.close(fig)
    log("saved plots/alignment_sessions.png")

In [ ]:
# ==================== Window extraction (index references) ===================
# Windows are REFERENCES into packed session files (session_id, start, end) --
# no tensor duplication. Geometry matches the live contract: 100 steps = 8 s
# (the live helium deque), 50% stride.
import pandas as pd

WINDOW_COLS = [
    "window_id", "session_id", "start_step", "end_step", "voice", "scenario_id",
    "intended_emotion", "user_clip", "user_clip_emotion",
    "vocal_label_id", "vocal_label", "vocal_conf", "vocal_scores_json",
    "semantic_label_id", "semantic_label", "semantic_conf", "semantic_scores_json", "semantic_source",
    "expressed_label_id", "expressed_label", "agreement", "review_needed", "corrected", "window_text",
]

_aligned_ok = set(align_df[align_df.ok].session_id.astype(str)) if "ok" in align_df.columns else set()
_sessions_for_windows = [s for s in _ok_sessions if s in _aligned_ok]
log("extracting windows from %d alignment-verified sessions" % len(_sessions_for_windows))

if WINDOWS_INDEX_CSV.exists():
    windows_df = pd.read_csv(WINDOWS_INDEX_CSV)
    log("windows_index.csv already exists (%d windows) -- skipping extraction (delete it to rebuild)" % len(windows_df))
else:
    _rows = []
    for _sid in _sessions_for_windows:
        _s = load_session(session_paths(_sid))
        T = int(_s["helium"].shape[0])
        _m = _s["meta"]
        _w = 0
        for start in range(0, max(1, T - WINDOW_STEPS + 1), WINDOW_STRIDE):
            end = min(T, start + WINDOW_STEPS)
            if end - start < WINDOW_STEPS // 2:
                continue
            _wtoks = _s["text_tokens"][start:end].tolist()
            _rows.append({
                "window_id": "%s_w%03d" % (_sid, _w), "session_id": _sid,
                "start_step": int(start), "end_step": int(end),
                "voice": _m["voice"], "scenario_id": _m["scenario_id"],
                "intended_emotion": _m["intended_emotion"],
                "user_clip": _m["user_clip"], "user_clip_emotion": _m["user_clip_emotion"],
                "window_text": detokenize_text_tokens(_wtoks),
            })
            _w += 1
    windows_df = pd.DataFrame(_rows, columns=WINDOW_COLS)
    windows_df.to_csv(WINDOWS_INDEX_CSV, index=False)
    log("extracted %d windows -> %s" % (len(windows_df), WINDOWS_INDEX_CSV))
print(windows_df.groupby(["voice", "scenario_id"]).size().unstack(fill_value=0).to_string())

## Dual-teacher window labeling

- **Vocal teacher** (emotion2vec) runs on each window's **reply audio slice** -- how the window *sounds*.
- **Semantic teacher** (text classifier) runs on the window's **detokenized text stream** -- the model's own words are an exact transcript (no ASR needed for the primary path; windows with too little text inherit the session-level semantic label and are flagged).

Both label sets + full score vectors are preserved per window. Fusion + human review follow in the next cell.

In [ ]:
# ==================== Dual-teacher window labeling (resumable) ===============
from auto_emotion import SpeechEmotionClassifier, SER_LABEL_TO_EMOTION_ID
from tqdm.auto import tqdm

windows_df = pd.read_csv(WINDOWS_INDEX_CSV)
_need = windows_df[windows_df.vocal_label_id.isna()]
log("labeling %d windows (vocal + semantic)" % len(_need))

if len(_need):
    _audio_teacher = SpeechEmotionClassifier(AUDIO_TEACHER_MODEL, device=TEACHER_DEVICE if torch.cuda.is_available() else "cpu", half=False, disgust_map=DISGUST_MAP)
    import torchaudio

    def _english_part(label):
        for p in str(label).split("/"):
            p = p.strip()
            if p and all(ord(c) < 128 for c in p):
                return p.lower()
        return str(label).lower()

    E2V_TO_ID = {"angry": 3, "anger": 3, "happy": 1, "sad": 2, "neutral": 0, "surprised": 4, "surprise": 4, "fearful": 5, "fear": 5, "disgusted": DISGUST_MAP}
    TEXT_TO_ID = {"anger": 3, "joy": 1, "neutral": 0, "sadness": 2, "surprise": 4, "fear": 5, "disgust": DISGUST_MAP}

    def audio_scores_6(pcm24):
        wav = torchaudio.functional.resample(torch.from_numpy(np.asarray(pcm24, np.float32).reshape(1, -1)), TARGET_SR, 16000)
        try:
            res = _audio_teacher.model.generate(wav[0].numpy(), fs=16000, granularity="utterance", extract_embedding=False)
        except Exception:
            import soundfile as _sf, tempfile as _tmp
            with _tmp.NamedTemporaryFile(suffix=".wav", delete=False) as f:
                _sf.write(f.name, wav[0].numpy(), 16000); tmp = f.name
            try:
                res = _audio_teacher.model.generate(tmp, fs=16000, granularity="utterance", extract_embedding=False)
            finally:
                os.unlink(tmp)
        r0 = res[0]
        out = np.zeros(NUM_EMOTIONS)
        for lab, sc in zip(r0.get("labels") or [], r0.get("scores") or []):
            eid = E2V_TO_ID.get(_english_part(lab))
            if eid is not None:
                out[eid] = max(out[eid], float(sc))
        s = out.sum()
        return out / s if s > 0 else out + 1.0 / NUM_EMOTIONS

    _text_teacher = None
    if RUN_TEXT_TEACHER:
        from transformers import pipeline as _hf_pipeline
        _text_teacher = _hf_pipeline("text-classification", model=TEXT_TEACHER_MODEL, top_k=None,
                                     device=0 if (TEACHER_DEVICE == "cuda" and torch.cuda.is_available()) else -1)

    def text_scores_6(text):
        if _text_teacher is None or not text or len(text.split()) < 2:
            return None
        out = _text_teacher(text[:512])
        rows = out[0] if isinstance(out, list) and out and isinstance(out[0], list) else out
        v = np.zeros(NUM_EMOTIONS)
        for r in rows:
            eid = TEXT_TO_ID.get(str(r["label"]).lower())
            if eid is not None:
                v[eid] = max(v[eid], float(r["score"]))
        s = v.sum()
        return v / s if s > 0 else None

    _session_cache = {}
    def session_for(sid):
        if sid not in _session_cache:
            if len(_session_cache) > 4:
                _session_cache.clear()
            _session_cache[sid] = load_session(session_paths(sid))
        return _session_cache[sid]

    _flush = 0
    for _i, _row in tqdm(_need.iterrows(), total=len(_need), desc="label windows"):
        _s = session_for(str(_row.session_id))
        _pcm = np.asarray(_s["reply_pcm"], np.float32)[int(_row.start_step) * FRAME_SIZE:int(_row.end_step) * FRAME_SIZE]
        _v6 = audio_scores_6(_pcm)
        _vid = int(_v6.argmax())
        _m = windows_df.index == _i
        windows_df.loc[_m, "vocal_label_id"] = _vid
        windows_df.loc[_m, "vocal_label"] = EMOTION_NAMES[_vid]
        windows_df.loc[_m, "vocal_conf"] = round(float(_v6[_vid]), 4)
        windows_df.loc[_m, "vocal_scores_json"] = json.dumps([round(float(x), 4) for x in _v6])
        _t6 = text_scores_6(str(_row.window_text) if isinstance(_row.window_text, str) else "")
        _src = "detok"
        if _t6 is None:
            _sess_text = _s["meta"].get("detok_text", "")
            _t6 = text_scores_6(_sess_text)
            _src = "session_inherited" if _t6 is not None else "none"
        if _t6 is not None:
            _tid = int(_t6.argmax())
            windows_df.loc[_m, "semantic_label_id"] = _tid
            windows_df.loc[_m, "semantic_label"] = EMOTION_NAMES[_tid]
            windows_df.loc[_m, "semantic_conf"] = round(float(_t6[_tid]), 4)
            windows_df.loc[_m, "semantic_scores_json"] = json.dumps([round(float(x), 4) for x in _t6])
        windows_df.loc[_m, "semantic_source"] = _src
        _flush += 1
        if _flush % 100 == 0:
            windows_df.to_csv(WINDOWS_INDEX_CSV, index=False)
            log("  flushed %d/%d" % (_flush, len(_need)))
    windows_df.to_csv(WINDOWS_INDEX_CSV, index=False)

log("window labeling complete.")
print("vocal distribution:")
print(windows_df.vocal_label.value_counts(dropna=False).to_string())
print("semantic sources: %s" % windows_df.semantic_source.value_counts(dropna=False).to_dict())

In [ ]:
# ============== Fusion + human review + corrections (idempotent) =============
FUSION_POLICY = "vocal_wins"   # "vocal_wins" | "semantic_wins" | "confidence" | "drop"
LOW_CONF = 0.5

windows_df = pd.read_csv(WINDOWS_INDEX_CSV)
_has_sem = windows_df.semantic_label_id.notna()
_agree = _has_sem & (windows_df.vocal_label_id == windows_df.semantic_label_id)
windows_df["agreement"] = np.where(~_has_sem, "no_semantic", np.where(_agree, "agree", "disagree"))
windows_df["review_needed"] = False

def _fuse(row):
    v = int(row.vocal_label_id)
    if pd.isna(row.semantic_label_id):
        return v, float(row.vocal_conf) < LOW_CONF
    s = int(row.semantic_label_id)
    if v == s:
        return v, ((float(row.vocal_conf) + float(row.semantic_conf)) / 2.0) < LOW_CONF
    if FUSION_POLICY == "semantic_wins":
        return s, True
    if FUSION_POLICY == "confidence":
        return (v if float(row.vocal_conf) >= float(row.semantic_conf) else s), True
    if FUSION_POLICY == "drop":
        return -1, True
    return v, True

_fused = windows_df.apply(_fuse, axis=1, result_type="expand")
windows_df["expressed_label_id"] = _fused[0].astype(int)
windows_df["review_needed"] = _fused[1].astype(bool)
windows_df["expressed_label"] = windows_df.expressed_label_id.map(lambda i: EMOTION_NAMES.get(int(i), "dropped"))
windows_df["corrected"] = windows_df["corrected"].fillna(False)

def apply_corrections(path=CORRECTIONS_CSV):
    if not Path(path).exists():
        return 0
    corr = pd.read_csv(path)
    n = 0
    for _, r in corr.iterrows():
        m = windows_df.window_id.astype(str) == str(r.window_id)
        if not m.any():
            continue
        for col, idcol in (("expressed_label", "expressed_label_id"), ("vocal_label", "vocal_label_id"), ("semantic_label", "semantic_label_id")):
            val = r.get(col)
            if isinstance(val, str) and val.strip().lower() in EMOTION_IDS:
                windows_df.loc[m, col] = val.strip().lower()
                windows_df.loc[m, idcol] = EMOTION_IDS[val.strip().lower()]
        windows_df.loc[m, "review_needed"] = False
        windows_df.loc[m, "corrected"] = True
        n += 1
    log("applied %d corrections" % n)
    return n

apply_corrections()
windows_df.to_csv(WINDOWS_INDEX_CSV, index=False)
windows_df[windows_df.review_needed].to_csv(REVIEW_QUEUE_CSV, index=False)
log("fusion done: agreement=%s | review queue=%d" % (windows_df.agreement.value_counts().to_dict(), int(windows_df.review_needed.sum())))
print("expressed distribution:")
print(windows_df.expressed_label.value_counts().to_string())
print()
print("intended -> expressed agreement (steering reliability; metadata only, never a label):")
_ie = windows_df[windows_df.expressed_label_id >= 0]
print((_ie.intended_emotion == _ie.expressed_label).value_counts(normalize=True).round(3).to_string())
print()
print("Human review: open review_queue.csv, listen to sessions/<session_id>.pt reply_pcm")
print("(or use the playback cell below), fill corrections.csv (window_id, expressed_label, ...), re-run this cell.")

In [ ]:
# ===================== QC + audio spot-check + manifest ======================
# Listen to a few windows directly from the packed sessions (sanity check that
# labels match what you hear), then write the Phase 2 manifest.
from IPython.display import Audio, display

def window_pcm(window_id):
    r = windows_df[windows_df.window_id == window_id].iloc[0]
    s = load_session(session_paths(str(r.session_id)))
    return np.asarray(s["reply_pcm"], np.float32)[int(r.start_step) * FRAME_SIZE:int(r.end_step) * FRAME_SIZE], r

_review_samples = windows_df[windows_df.expressed_label_id >= 0].groupby("expressed_label").head(1)
for _, _r in _review_samples.iterrows():
    _pcm, _ = window_pcm(_r.window_id)
    print("%s | intended=%s vocal=%s(%.2f) semantic=%s | %s" % (
        _r.window_id, _r.intended_emotion, _r.vocal_label, _r.vocal_conf,
        _r.semantic_label if pd.notna(_r.semantic_label) else "-",
        str(_r.window_text)[:80]))
    display(Audio(_pcm, rate=TARGET_SR))

# class balance + leakage groups
windows_df = pd.read_csv(WINDOWS_INDEX_CSV)
_trainable = windows_df[windows_df.expressed_label_id >= 0]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
_trainable.expressed_label.value_counts().reindex(list(EMOTION_NAMES.values())).plot.bar(ax=axes[0], title="expressed label balance")
_trainable.groupby("voice").size().plot.bar(ax=axes[1], title="windows per voice")
_trainable.groupby("scenario_id").size().plot.bar(ax=axes[2], title="windows per scenario")
fig.tight_layout(); fig.savefig(str(PLOTS_DIR / "dataset_balance.png"), dpi=110); plt.close(fig)
log("saved plots/dataset_balance.png")

manifest = {
    "phase": 1,
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "repo_commit": REPO_COMMIT,
    "contract": {"hidden_dim": 4096, "step_hz": STEP_HZ, "frame_size": FRAME_SIZE, "sample_rate": TARGET_SR,
                 "window_steps": WINDOW_STEPS, "window_stride": WINDOW_STRIDE, "lookahead_steps": LOOKAHEAD_STEPS,
                 "num_codebooks": NUM_CODEBOOKS, "quantize_4bit": QUANTIZE_4BIT, "cfg_coef": CFG_COEF,
                 "num_emotions": NUM_EMOTIONS, "emotion_names": EMOTION_NAMES},
    "engine": {"moshi_root": str(BNB4_DIR), "bnb4_hf_repo": BNB4_HF_REPO, "gated_hf_repo": GATED_HF_REPO,
               "capture": "liveTry.MoshiOnlyEngine subclass; lm_gen._step graph-compatible capture (layer[-2])"},
    "counts": {"sessions_ok": int((sessions_index.status == "ok").sum()),
               "sessions_total": int(len(sessions_index)),
               "windows_total": int(len(windows_df)),
               "windows_trainable": int(len(_trainable)),
               "review_pending": int(windows_df.review_needed.sum()),
               "alignment_ok": int(align_df.ok.sum()) if "ok" in align_df.columns else 0},
    "label_policy": {"teachers": [AUDIO_TEACHER_MODEL, TEXT_TEACHER_MODEL if RUN_TEXT_TEACHER else None],
                     "fusion_policy": FUSION_POLICY, "low_conf": LOW_CONF,
                     "intended_emotion_is_metadata_only": True},
    "leakage_groups": {"voice": sorted(_trainable.voice.unique().tolist()),
                       "session": "session_id", "note": "split by session_id (and voice for generalization tests)"},
    "paths": {"sessions_dir": str(SESSIONS_DIR), "windows_index": str(WINDOWS_INDEX_CSV),
              "sessions_index": str(SESSIONS_INDEX_CSV), "alignment": str(ALIGNMENT_CSV),
              "review_queue": str(REVIEW_QUEUE_CSV), "corrections": str(CORRECTIONS_CSV)},
    "phase0_recommendation": (json.loads((PHASE0_DIR / "results" / "phase0_results.json").read_text())["recommendation"]
                              if (PHASE0_DIR / "results" / "phase0_results.json").exists() else None),
}
save_json(manifest, MANIFEST_JSON)
log("wrote %s" % MANIFEST_JSON)
print(json.dumps(manifest["counts"], indent=2))

## Phase 1 complete

The dataset is ready for **Phase 2** (`Phase_2_Train_Unified_PersonaPlex_Emotion_Network.ipynb`):

- `phase1_dataset.json` -- manifest (contract, counts, leakage groups, Phase 0 recommendation)
- `windows_index.csv` -- per-window dual labels (vocal + semantic, kept separate), expressed labels, confidences, agreement/review flags, voice/scenario/session groups
- `sessions/*.pt` -- packed per-step Helium hidden states [T,4096] fp16, codes [8,T], text tokens, reply PCM

**Before Phase 2:** clear the review queue if you can (fill `corrections.csv` and re-run the fusion cell). The more expressed labels are human-verified, the better the unified head. Phase 2 splits by `session_id` (and reports a voice-held-out generalization score) so there is no leakage.